In [ ]:
import ipywidgets as widgets
from IPython.display import display

samples = []

role_tabs = widgets.Tab()
manager_panel = widgets.VBox()
supplier_panel = widgets.VBox()
role_tabs.children = (manager_panel, supplier_panel)
role_tabs.set_title(0, "Utilisateur 1")
role_tabs.set_title(1, "Fournisseur")

reference_input = widgets.Text(
    description="Référence",
    placeholder="Ex. SAMPLE-002",
    layout=widgets.Layout(width="420px"),
)
create_button = widgets.Button(description="Créer la référence", button_style="success")
manager_output = widgets.Output()

sample_selector = widgets.Dropdown(description="Sample", layout=widgets.Layout(width="420px"))
request_date = widgets.DatePicker(description="Date de demande")
availability = widgets.Dropdown(
    description="Disponible ?",
    options=[("À renseigner", "À renseigner"), ("YES", "YES"), ("NO", "NO")],
    value="À renseigner",
)
availability_date = widgets.DatePicker(description="Date de disponibilité")
ship_date = widgets.DatePicker(description="Date d'envoi")
receipt_date = widgets.DatePicker(description="Date de réception")
returned = widgets.Dropdown(description="Renvoyé ?", options=["NO", "YES"], value="NO")
return_date = widgets.DatePicker(description="Date de renvoi")
supplier_output = widgets.Output()

availability_date.layout.display = "none"
return_date.layout.display = "none"

def render_manager():
    with manager_output:
        manager_output.clear_output(wait=True)
        if not samples:
            print("Aucune référence créée pour le moment.")
            return
        for sample in samples:
            print(f"\n{sample['reference']}")
            print(f"  Demande : {sample['dateDemande'] or 'En attente'}")
            print(f"  Disponible : {sample['disponible'] or 'À renseigner'}")
            if sample["disponible"] == "NO":
                print(f"  Disponibilité prévue : {sample['dateDisponibilite'] or 'À renseigner'}")
            print(f"  Envoi : {sample['dateEnvoi'] or 'En attente'}")
            print(f"  Réception : {sample['dateReception'] or 'En attente'}")
            print(f"  Renvoyé : {sample['renvoye']}")
            if sample["renvoye"] == "YES":
                print(f"  Date de renvoi : {sample['dateRenvoi'] or 'À renseigner'}")

def refresh_selector():
    previous = sample_selector.value
    sample_selector.options = [(sample["reference"], sample["reference"]) for sample in samples]
    if previous in [sample["reference"] for sample in samples]:
        sample_selector.value = previous
    elif samples:
        sample_selector.value = samples[0]["reference"]
    else:
        sample_selector.value = None

def create_reference(_):
    value = reference_input.value.strip()
    if not value:
        return
    if any(sample["reference"].casefold() == value.casefold() for sample in samples):
        reference_input.value = ""
        with manager_output:
            manager_output.clear_output(wait=True)
            print("Cette référence existe déjà.")
        return
    samples.append({
        "reference": value,
        "dateDemande": None,
        "disponible": "",
        "dateDisponibilite": None,
        "dateEnvoi": None,
        "dateReception": None,
        "renvoye": "NO",
        "dateRenvoi": None,
    })
    reference_input.value = ""
    refresh_selector()
    render_manager()

create_button.on_click(create_reference)

def save_supplier_fields(change=None):
    reference = sample_selector.value
    sample = next((item for item in samples if item["reference"] == reference), None)
    if sample is None:
        return
    disponible = "" if availability.value == "À renseigner" else availability.value
    sample.update({
        "dateDemande": request_date.value,
        "disponible": disponible,
        "dateDisponibilite": availability_date.value if disponible == "NO" else None,
        "dateEnvoi": ship_date.value,
        "dateReception": receipt_date.value,
        "renvoye": returned.value,
        "dateRenvoi": return_date.value if returned.value == "YES" else None,
    })
    availability_date.layout.display = "" if disponible == "NO" else "none"
    return_date.layout.display = "" if returned.value == "YES" else "none"
    render_manager()

def load_selected_sample(change=None):
    sample = next((item for item in samples if item["reference"] == sample_selector.value), None)
    if sample is None:
        return
    values = [
        (request_date, "dateDemande"),
        (availability, "disponible"),
        (availability_date, "dateDisponibilite"),
        (ship_date, "dateEnvoi"),
        (receipt_date, "dateReception"),
        (returned, "renvoye"),
        (return_date, "dateRenvoi"),
    ]
    for control, key in values:
        control.unobserve(save_supplier_fields, names="value")
        value = sample[key]
        if control is availability and not value:
            value = "À renseigner"
        control.value = value
        control.observe(save_supplier_fields, names="value")
    availability_date.layout.display = "" if sample["disponible"] == "NO" else "none"
    return_date.layout.display = "" if sample["renvoye"] == "YES" else "none"

sample_selector.observe(load_selected_sample, names="value")
for control in [request_date, availability, availability_date, ship_date, receipt_date, returned, return_date]:
    control.observe(save_supplier_fields, names="value")

manager_panel.children = (
    widgets.HTML("<h2>Créer une référence</h2><p>Le fournisseur pourra ensuite compléter son suivi.</p>"),
    widgets.HBox([reference_input, create_button]),
    manager_output,
)
supplier_panel.children = (
    widgets.HTML("<h2>Informations du sample</h2><p>Choisissez une référence puis renseignez les dates et statuts.</p>"),
    sample_selector,
    request_date,
    availability,
    availability_date,
    ship_date,
    receipt_date,
    returned,
    return_date,
    supplier_output,
)

notice = widgets.HTML(
    "<p style='padding:10px 12px;background:#fff4d6;border-left:3px solid #d5a339'>"
    "Prototype local : les données restent en mémoire dans cette session. "
    "Pour partager les modifications entre le gestionnaire et le fournisseur sur des appareils différents, "
    "il faudra connecter une base de données et un service d'authentification.</p>"
)
display(notice, role_tabs)

TraitError: Invalid selection: value not found